# CoTraffic - training job on Kaggle

One training run per session: choose the variant, network and seed below. Checkpoints are
written every 50 000 agent steps to `/kaggle/working`, which Kaggle keeps as the notebook
output. To continue an interrupted run, add that output as an input dataset and set
`RESUME_FROM` to its folder: training restarts from the latest checkpoint.

In [ ]:
AGENT = "coop"          # ind, coop, coop_eco05, coop_eco, coop_full
NET = "grid4x4"         # grid4x4 or real_district
SEED = 1
TIMESTEPS = None        # None = value of configs/agents/<AGENT>.yaml
RESUME_FROM = None      # e.g. "/kaggle/input/cotraffic-coop-s1/training"
REPO_URL = "https://github.com/<team>/cotraffic.git"   # or attach the repository as a dataset
REPO = "/kaggle/working/cotraffic"

In [ ]:
import os, shutil, subprocess
if not os.path.isdir(REPO):
    if os.path.isdir("/kaggle/input/cotraffic"):
        shutil.copytree("/kaggle/input/cotraffic", REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)

In [ ]:
import subprocess, sys
# SUMO and the RL stack; torch is already installed on the notebook image.
reqs = [l for l in open(f"{REPO}/requirements.txt") if l.strip() and not l.startswith(("#", "torch"))]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[r.split("#")[0].strip() for r in reqs]], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)

In [ ]:
import shutil, os
run = f"{REPO}/results/training/{AGENT}/{NET}/seed{SEED}"
if RESUME_FROM and not os.path.isdir(run):
    shutil.copytree(f"{RESUME_FROM}/{AGENT}/{NET}/seed{SEED}", run)
os.environ["LIBSUMO_AS_TRACI"] = "1"
args = ["python", "-m", "cotraffic", "train", "--agent", AGENT, "--net", NET, "--seed", str(SEED)]
if TIMESTEPS:
    args += ["--timesteps", str(TIMESTEPS)]
!cd {REPO} && {" ".join(args)}

In [ ]:
# Keep the run (checkpoints, episode log, run_info.json) as notebook output.
out = f"/kaggle/working/training/{AGENT}/{NET}/seed{SEED}"
shutil.copytree(run, out, dirs_exist_ok=True, ignore=shutil.ignore_patterns("tensorboard"))
print(open(f"{out}/run_info.json").read()[:2000])